# Missing Data Is Not Empty Space
## Statistical analysis of an irregular blood-pressure tracker

This executable notebook treats the **observation process** as part of the statistical problem. The public repository contains only a day-indexed aggregate snapshot; the original personal-health workbook is intentionally excluded. This is a statistical case study, not a clinical interpretation.

## 1. Load the explicit calendar grid

Missing calendar days remain rows with no measurements. We do not silently interpolate them away.

In [ ]:
from pathlib import Path
import csv, json
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
from scipy import stats

root=Path.cwd(); data=root/"data/analysis_snapshot.csv"; audit_path=root/"data/source_audit.json"
if not data.exists(): raise FileNotFoundError("Run from Blood_Pressure_Missingness/")
with data.open(newline="",encoding="utf-8") as f: rows=list(csv.DictReader(f))
with audit_path.open(encoding="utf-8") as f: audit=json.load(f)
idx=np.array([int(r["day_index"]) for r in rows]); obs=np.array([r["observed"]=="1" for r in rows])
counts=np.array([int(r["n_readings"]) for r in rows]); syst=np.array([float(r["mean_systolic_mmHg"]) if r["mean_systolic_mmHg"] else np.nan for r in rows])
if not np.array_equal(idx,np.arange(len(rows))): raise ValueError("Calendar index is not contiguous")
print(f"{len(rows)} calendar days, {obs.sum()} observed, {(~obs).sum()} missing")
print(f"Valid readings: {audit['valid_measurements']}; sessions: {audit['sessionization']['n_sessions']}")

## 2. Missingness geometry

There are **25 observed days across 64 calendar days**, so **60.9% of days are missing**. One **32-day uninterrupted gap** accounts for **82.1% of all missing days**. This is not a setting where a straight interpolation line should be mistaken for data.

In [ ]:
runs=[]; start=None
for i,seen in enumerate(obs):
    if not seen and start is None: start=i
    if seen and start is not None: runs.append((start,i-1,i-start)); start=None
if start is not None: runs.append((start,len(obs)-1,len(obs)-start))
print("Missing runs:",runs); print("Longest gap:",max(x[2] for x in runs),"days")
fig,ax=plt.subplots(figsize=(10,4)); ax.bar(idx,counts); ax.set(xlabel="Day index",ylabel="Valid readings",title="Sampling intensity is highly uneven"); plt.show()

## 3. Unequal sampling changes the estimand

Treating all 144 readings as i.i.d. overweights days with many measurements. We compare the reading-weighted mean with the equal-observed-day mean and test whether sampling intensity is associated with the observed daily mean.

In [ ]:
x=counts[obs].astype(float); y=syst[obs]
weighted=float(np.average(y,weights=x)); equal=float(np.mean(y)); p=stats.pearsonr(x,y); s=stats.spearmanr(x,y)
print(f"Systolic reading-weighted mean: {weighted:.2f} mmHg")
print(f"Systolic equal-day mean:        {equal:.2f} mmHg")
print(f"Difference:                     {weighted-equal:.2f} mmHg")
print(f"Pearson r={p.statistic:.3f}, p={p.pvalue:.3f}; Spearman rho={s.statistic:.3f}, p={s.pvalue:.3f}")
coef=np.polyfit(x,y,1); g=np.linspace(x.min(),x.max(),100)
fig,ax=plt.subplots(figsize=(7,4.5)); ax.scatter(x,y); ax.plot(g,coef[1]+coef[0]*g); ax.set(xlabel="Readings on observed day",ylabel="Daily mean systolic (mmHg)",title="Sampling intensity is associated with the observed mean"); plt.show()

The association is negative (**Pearson r ≈ −0.485, p ≈ 0.014**), and the reading-weighted systolic mean (**115.85 mmHg**) is lower than the equal-day mean (**118.27 mmHg**). The p-value is secondary here; structurally, the observation process is not behaving as if it were irrelevant.

The workbook audit also found six blank placeholder rows whose derived pulse-pressure value is zero. They lower the spreadsheet pulse-pressure mean from **40.17** to **38.57 mmHg**, a **4% downward bias**. `Meal` and `Symptoms` are unrecorded in 83.3% and 89.6% of valid rows respectively, so blanks should not be recoded as negative labels without a collection rule.

## 4. Robust trend and state-space uncertainty

A simple day-level trend is descriptive only. HC3 standard errors reduce sensitivity to heteroskedasticity. For the long calendar gaps, a local-linear-trend state-space model leaves missing observations missing while propagating uncertainty through the latent state.

In [ ]:
t=idx[obs].astype(float); X=sm.add_constant(t); fit=sm.OLS(y,X).fit(cov_type="HC3"); ci=np.asarray(fit.conf_int())[1]*30
print(f"Systolic trend: {fit.params[1]*30:.2f} mmHg/30d, 95% CI [{ci[0]:.2f}, {ci[1]:.2f}], p={fit.pvalues[1]:.4f}")
model=sm.tsa.UnobservedComponents(syst,level="local linear trend"); res=model.fit(disp=False)
level=np.asarray(res.smoothed_state[0]); var=np.asarray(res.smoothed_state_cov[0,0,:]); sd=np.sqrt(np.maximum(var,0))
fig,ax=plt.subplots(figsize=(10,4.5)); ax.scatter(t,y,label="Observed daily mean"); ax.plot(idx,level,label="Smoothed latent level"); ax.fill_between(idx,level-1.96*sd,level+1.96*sd,alpha=.2,label="95% state interval"); ax.set(xlabel="Day index",ylabel="Systolic (mmHg)",title="State-space model carries uncertainty across missing days"); ax.legend(); plt.show()

## 5. Statistical conclusion

The data do **not** identify the missingness mechanism as MCAR, MAR, or MNAR. What they do show is enough to reject a naïve flat analysis: low calendar coverage, one dominant long gap, strong variation in sampling intensity, and an association between sampling intensity and observed pressure.

A defensible workflow is **audit → sessionise → equal-day summaries → explicit missing calendar grid → robust trend/state-space model → uncertainty, not invented observations**.